# 14 — Sistemas híbridos e reclassificação

**DATA SCIENCE KNOWLEDGE BASE**  
**Área:** `09_recommender_systems` — Sistemas de Recomendação  
**Capítulo:** 14 de 16  

> Material didático autoral revisado a partir dos notebooks de estudo. Execute as células em sequência, de cima para baixo, em um kernel reiniciado. Consulte as fontes históricas somente para estudo, preservando sua proveniência.

## Objetivos de aprendizagem
- Combinar conteúdo e popularidade
- Normalizar escalas e filtrar itens já avaliados

## Materiais complementares

- [Guia Técnico](reference_materials/14_sistemas_hibridos.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_14_sistemas_hibridos.pdf)

**Navegação:** os links são relativos à pasta deste notebook. A versão editorial desses PDFs será revalidada após a consolidação científica dos notebooks.

## Fundamentação conceitual
Sistemas híbridos combinam sinais complementares. Pontuações de escalas distintas precisam de normalização. Os pesos devem ser escolhidos em validação, evitando ajuste no teste.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Sinais de conteúdo e popularidade

In [2]:
from sklearn.preprocessing import MultiLabelBinarizer
X=MultiLabelBinarizer().fit_transform(movies.genres.str.split('|'))
position=movies.index[movies.movieId.eq(1)][0]
content=cosine_similarity(X[position:position+1],X).ravel()
counts=ratings.movieId.value_counts();pop=movies.movieId.map(counts).fillna(0).to_numpy()
pop=np.log1p(pop);pop=pop/(pop.max() or 1)
print('Faixas:',(content.min(),content.max()),(pop.min(),pop.max()))

Faixas: (np.float64(0.0), np.float64(1.0)) (np.float64(0.0), np.float64(1.0))


**Interpretação.** Popularidade recebe log para reduzir domínio da cauda alta.

## Combinação linear

In [3]:
alpha=.7
combined=alpha*content+(1-alpha)*pop
seen=set(ratings.loc[ratings.userId.eq(1),'movieId']); eligible=~movies.movieId.isin(seen)
rank=movies.assign(score=combined)[eligible].sort_values('score',ascending=False).head(10)
print(rank[['title','genres','score']].to_string(index=False))

                           title                                              genres    score
           Monsters, Inc. (2001)         Adventure|Animation|Children|Comedy|Fantasy 0.952989
              Toy Story 2 (1999)         Adventure|Animation|Children|Comedy|Fantasy 0.937191
                    Shrek (2001) Adventure|Animation|Children|Comedy|Fantasy|Romance 0.904999
                     Antz (1998)         Adventure|Animation|Children|Comedy|Fantasy 0.898064
Emperor's New Groove, The (2000)         Adventure|Animation|Children|Comedy|Fantasy 0.888180
             Finding Nemo (2003)                 Adventure|Animation|Children|Comedy 0.882475
            Bug's Life, A (1998)                 Adventure|Animation|Children|Comedy 0.860581
          Shrek the Third (2007)         Adventure|Animation|Children|Comedy|Fantasy 0.859907
                  Ice Age (2002)                 Adventure|Animation|Children|Comedy 0.856532
              Toy Story 3 (2010)    Adventure|Animation|Chil

**Interpretação.** Pesos diferentes podem trocar relevância semântica por popularidade.

## Sensibilidade dos pesos

In [4]:
for w in [0,.5,1]:
    order=np.argsort(-(w*content+(1-w)*pop))[:3]
    print(w,movies.iloc[order].title.tolist())

0 ['Forrest Gump (1994)', 'Shawshank Redemption, The (1994)', 'Pulp Fiction (1994)']
0.5 ['Toy Story (1995)', 'Monsters, Inc. (2001)', 'Shrek (2001)']
1 ['Toy Story (1995)', 'Shrek the Third (2007)', 'Antz (1998)']


**Interpretação.** O teste de sensibilidade é exploratório, não substitui validação de ranking.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** fusão de escores. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **fusão de escores** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **fusão de escores** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — Evitar vazamento em escores híbridos

No exemplo original, a popularidade é calculada usando o histórico completo e o peso `alpha` é arbitrário. O experimento é válido para exploração, mas **não é avaliação fora da amostra**. Para validá-lo, recorte o treino antes de construir os sinais, selecione pesos na validação e reserve o teste final.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [5]:
# Ilustração sem vazamento futuro: popularidade obtida somente do passado
corte = ratings.timestamp.quantile(.8)
historico_treino = ratings.loc[ratings.timestamp <= corte]
contagens_treino = historico_treino.movieId.value_counts()
pop_treino = np.log1p(movies.movieId.map(contagens_treino).fillna(0).to_numpy())
pop_treino /= (pop_treino.max() or 1)
score_treino = .7*content + .3*pop_treino
print('10 candidatos por escores construídos do treino:',movies.assign(score=score_treino).loc[~movies.movieId.isin(set(historico_treino.loc[historico_treino.userId.eq(1),'movieId']))].nlargest(10,'score').title.tolist())
print('A lista acima NÃO é uma métrica de generalização.')

10 candidatos por escores construídos do treino: ['Monsters, Inc. (2001)', 'Toy Story 2 (1999)', 'Shrek (2001)', 'Antz (1998)', 'Finding Nemo (2003)', "Emperor's New Groove, The (2000)", "Bug's Life, A (1998)", 'Shrek the Third (2007)', 'Ice Age (2002)', 'Aladdin (1992)']
A lista acima NÃO é uma métrica de generalização.


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

## Limitações e boas práticas
Esse híbrido serve à demonstração, não é resultado validado de produto.

## Exercícios
Quais cuidados seriam necessários para personalizar o conteúdo pelo histórico do usuário?

## Soluções comentadas
Construa um perfil médio dos filmes positivos de treino e exclua itens já vistos; não utilize notas do teste.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# EP_OUTPUTS_09_V1
# Exportação após os experimentos. Executar este notebook desde o início (Run All).
# Arquivos com prefixo do capítulo evitam colisões com outras saídas.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
rank[["movieId", "title", "genres", "score"]].to_csv(
    out / "14_ranking_hibrido_demonstrativo.csv", index=False)
print('Outputs do capítulo 14 salvos em:', out.resolve())


Outputs do capítulo 14 salvos em: /run/media/eduardo/Backup/Faculdades/PUC - CIENCIA DE DADOS/PUC/02_Disciplinas/09_recommender_systems/outputs
